# 08 — Recovery of Confirmed Planets (Leakage-Controlled)

## Research question
Among catalog entries labeled **CONFIRMED** or **FALSE POSITIVE**, what fraction of known confirmed entries receive a high confirmed-class probability when predictions are made on host stars excluded from model training?

This notebook uses **grouped out-of-fold (OOF) predictions**. Each host star is held out in exactly one fold, so all entries around a given star are kept together.

### Important interpretation
The reported “recovery” is a held-out **sensitivity/recall** for catalog-confirmed entries. It is not a measurement of the number of undiscovered planets recovered, and it does not establish a corrected planet occurrence rate.

Candidates are excluded from training and evaluation because their labels are unresolved. Disposition-derived columns (for example, `koi_score` and Robovetter false-positive flags) are excluded from model features to reduce label leakage.

In [1]:
# 1. Imports and reproducible settings

import os
import glob
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLDS = [0.30, 0.50, 0.70]

# Use the known Kaggle path when available; otherwise search Kaggle inputs.
DATA_PATH = "/kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv"

if not os.path.exists(DATA_PATH):
    csv_matches = sorted(
        p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
        if "cumulative" in os.path.basename(p).lower()
    )
    if len(csv_matches) == 0:
        raise FileNotFoundError(
            "Could not find the KOI cumulative CSV under /kaggle/input. "
            "Set DATA_PATH manually to the attached dataset."
        )
    if len(csv_matches) > 1:
        print("Multiple cumulative CSV files found:")
        for p in csv_matches:
            print(" -", p)
        print("\nUsing the first sorted match. Change DATA_PATH above if needed.")
    DATA_PATH = csv_matches[0]

print("Dataset:", DATA_PATH)

Dataset: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv


In [2]:
# 2. Load data and define the labeled evaluation sample

raw = pd.read_csv(DATA_PATH, comment="#")
print("Original rows:", len(raw))

required_cols = {"koi_disposition", "kepid", "koi_prad"}
missing_required = required_cols - set(raw.columns)
if missing_required:
    raise KeyError(f"Missing required columns: {sorted(missing_required)}")

# Only resolved catalog dispositions are used as supervised labels.
# CANDIDATE entries remain excluded because their outcome is unresolved.
df = raw.loc[
    raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
].copy()

# Preserve the original notebook's sample definition: require a host-star ID
# and a measured planet radius. Other feature missingness is handled in-pipeline.
df = df.dropna(subset=["kepid", "koi_prad"]).copy()
df = df.reset_index(drop=True)

df["label"] = (df["koi_disposition"] == "CONFIRMED").astype(int)
df["group"] = df["kepid"]

print("Resolved rows used:", len(df))
print("Unique host stars:", df["group"].nunique())
print("Confirmed rows:", int((df["label"] == 1).sum()))
print("False-positive rows:", int((df["label"] == 0).sum()))
print("Candidate rows excluded:", int((raw["koi_disposition"] == "CANDIDATE").sum()))

if df["label"].nunique() != 2:
    raise ValueError("Both CONFIRMED and FALSE POSITIVE classes must be present.")
if df["group"].nunique() < N_SPLITS:
    raise ValueError(
        f"Need at least {N_SPLITS} unique host stars; found {df['group'].nunique()}."
    )

Original rows: 9564
Resolved rows used: 7328
Unique host stars: 6424
Confirmed rows: 2746
False-positive rows: 4582
Candidate rows excluded: 1977


## 3. Feature selection and leakage controls

Only pre-disposition physical/catalog-observation measurements are eligible. The feature list is an explicit allow-list rather than “all numeric columns.”

The model deliberately excludes disposition labels, disposition-adjacent scores, vetting flags, and identifiers. This reduces the risk that the model simply learns the catalog's existing decision process.

In [3]:
# Explicit allow-list of numeric observational / physical features.
# Missing columns in a particular catalog release are simply skipped.
candidate_features = [
    "koi_period",
    "koi_time0bk",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_smet",
    "koi_srad",
    "koi_smass",
    "koi_kepmag",
    "koi_ror",
    "koi_max_mult_ev",
    "koi_num_transits"
]

feature_cols = [
    c for c in candidate_features
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]

if len(feature_cols) < 3:
    raise ValueError(
        "Fewer than 3 approved numeric features were found. "
        f"Available approved features: {feature_cols}"
    )

# Explicit audit: these columns must never be model inputs.
forbidden_tokens = [
    "disposition", "koi_score", "fpflag", "comment",
    "kepid", "kic", "label", "group"
]
leakage_hits = [
    c for c in feature_cols
    if any(token in c.lower() for token in forbidden_tokens)
]
if leakage_hits:
    raise AssertionError(f"Potential leakage columns entered features: {leakage_hits}")

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
y = df["label"].astype(int)
groups = df["group"]

print("Approved model features:")
for c in feature_cols:
    print(" -", c)
print("\nFeature count:", len(feature_cols))
print("Missing values are imputed separately inside each training fold.")

Approved model features:
 - koi_period
 - koi_time0bk
 - koi_impact
 - koi_duration
 - koi_depth
 - koi_prad
 - koi_teq
 - koi_insol
 - koi_model_snr
 - koi_steff
 - koi_slogg
 - koi_srad
 - koi_kepmag

Feature count: 13
Missing values are imputed separately inside each training fold.


## 4. Host-star-grouped out-of-fold predictions

`GroupKFold` assigns every row to exactly one test fold while keeping all rows from the same `kepid` together. The imputer and classifier are fitted only on each fold's training rows.

The OOF array starts as missing values. The notebook asserts that every row is filled exactly once; this prevents unscored rows from silently being treated as probability zero or being overwritten by repeated random splits.

In [ ]:
# 5-fold GroupKFold: every row is test data exactly once.
n_splits = min(N_SPLITS, groups.nunique())
gkf = GroupKFold(n_splits=n_splits)

oof_probability = np.full(len(df), np.nan, dtype=float)
oof_fold = np.full(len(df), -1, dtype=int)
prediction_count = np.zeros(len(df), dtype=int)

fold_summaries = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
    train_groups = set(groups.iloc[train_idx].unique())
    test_groups = set(groups.iloc[test_idx].unique())

    overlap = train_groups.intersection(test_groups)
    if overlap:
        raise AssertionError(
            f"Host-star leakage in fold {fold}: {len(overlap)} groups overlap."
        )

    if y.iloc[train_idx].nunique() < 2:
        raise ValueError(f"Training fold {fold} contains only one class.")

    model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE + fold,
            n_jobs=-1
        ))
    ])

    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    positive_class_position = list(model.named_steps["classifier"].classes_).index(1)
    fold_prob = model.predict_proba(X.iloc[test_idx])[:, positive_class_position]

    oof_probability[test_idx] = fold_prob
    oof_fold[test_idx] = fold
    prediction_count[test_idx] += 1

    fold_summaries.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_host_stars": len(train_groups),
        "test_host_stars": len(test_groups),
        "confirmed_test": int((y.iloc[test_idx] == 1).sum()),
        "false_positive_test": int((y.iloc[test_idx] == 0).sum())
    })

# Strict coverage and leakage checks.
if not np.all(prediction_count == 1):
    values, counts = np.unique(prediction_count, return_counts=True)
    raise AssertionError(
        f"Each row must be predicted exactly once. Counts: {dict(zip(values, counts))}"
    )
if np.isnan(oof_probability).any():
    raise AssertionError("Some rows have missing OOF probabilities.")
if (oof_fold < 1).any():
    raise AssertionError("Some rows were not assigned to a test fold.")

df["oof_confirmed_probability"] = oof_probability
df["oof_fold"] = oof_fold

fold_summary_df = pd.DataFrame(fold_summaries)
display(fold_summary_df)

print("\nOOF coverage:", int((prediction_count == 1).sum()), "/", len(df))
print("All host-star groups were kept intact in each fold.")

## 5. Overall held-out performance

Metrics are calculated once across all out-of-fold predictions, not by averaging overlapping random splits. The positive class is `CONFIRMED`.

In [ ]:
# Calculate pooled OOF discrimination metrics.
if y.nunique() == 2:
    oof_roc_auc = roc_auc_score(y, oof_probability)
    oof_average_precision = average_precision_score(y, oof_probability)
else:
    oof_roc_auc = np.nan
    oof_average_precision = np.nan

threshold = 0.50
oof_pred_05 = (oof_probability >= threshold).astype(int)

metrics = {
    "n_rows": len(df),
    "n_host_stars": int(groups.nunique()),
    "n_confirmed": int((y == 1).sum()),
    "n_false_positive": int((y == 0).sum()),
    "n_features": len(feature_cols),
    "n_group_folds": n_splits,
    "oof_roc_auc": oof_roc_auc,
    "oof_average_precision": oof_average_precision,
    "balanced_accuracy_at_0.50": balanced_accuracy_score(y, oof_pred_05),
    "precision_at_0.50": precision_score(y, oof_pred_05, zero_division=0),
    "sensitivity_recall_at_0.50": recall_score(y, oof_pred_05, zero_division=0),
    "f1_at_0.50": f1_score(y, oof_pred_05, zero_division=0)
}

metrics_df = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in metrics.items()]
)
display(metrics_df)

print("\nConfusion matrix at threshold 0.50")
print("Rows = actual [FALSE POSITIVE, CONFIRMED]")
print("Cols = predicted [FALSE POSITIVE, CONFIRMED]")
print(confusion_matrix(y, oof_pred_05, labels=[0, 1]))

print("\nClassification report at threshold 0.50")
print(classification_report(
    y, oof_pred_05,
    labels=[0, 1],
    target_names=["FALSE POSITIVE", "CONFIRMED"],
    zero_division=0
))

## 6. Confirmed-planet recovery across probability thresholds

For each threshold, “recovered confirmed” means a held-out catalog-confirmed row has predicted confirmed probability at or above that threshold. This is sensitivity/recall, not an independent confirmation of planetary status.

In [ ]:
threshold_rows = []

confirmed_mask = (y.to_numpy() == 1)
false_positive_mask = (y.to_numpy() == 0)

for t in THRESHOLDS:
    predicted_confirmed = (oof_probability >= t)

    confirmed_total = int(confirmed_mask.sum())
    confirmed_recovered = int((predicted_confirmed & confirmed_mask).sum())
    fp_total = int(false_positive_mask.sum())
    fp_flagged_as_confirmed = int((predicted_confirmed & false_positive_mask).sum())

    threshold_rows.append({
        "threshold": t,
        "confirmed_total": confirmed_total,
        "confirmed_recovered": confirmed_recovered,
        "confirmed_recall": (
            confirmed_recovered / confirmed_total if confirmed_total else np.nan
        ),
        "false_positive_total": fp_total,
        "false_positives_scored_above_threshold": fp_flagged_as_confirmed,
        "false_positive_rate_above_threshold": (
            fp_flagged_as_confirmed / fp_total if fp_total else np.nan
        )
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

row_05 = threshold_df.loc[threshold_df["threshold"] == 0.50].iloc[0]
print(
    f"At threshold 0.50: {int(row_05['confirmed_recovered'])} of "
    f"{int(row_05['confirmed_total'])} held-out confirmed entries "
    f"were classified as confirmed "
    f"(recall = {row_05['confirmed_recall']:.3f})."
)

## 7. Export reproducible results

The row-level file preserves the actual label, host-star ID, fold assignment, and OOF probability. The summary files can be used in later analysis, but they should not be interpreted as an occurrence-rate correction.

In [ ]:
# Row-level OOF predictions for auditing and later analysis.
id_cols = [
    c for c in ["kepid", "kepoi_name", " kepoi_name", "koi_disposition",
                "koi_period", "koi_prad"]
    if c in df.columns
]

oof_export = df[id_cols].copy()
oof_export["actual_label_confirmed"] = y.to_numpy()
oof_export["oof_confirmed_probability"] = oof_probability
oof_export["oof_fold"] = oof_fold

oof_export.to_csv("path2_recovery_oof_predictions.csv", index=False)
metrics_df.to_csv("path2_recovery_metrics.csv", index=False)
threshold_df.to_csv("path2_recovery_thresholds.csv", index=False)
fold_summary_df.to_csv("path2_recovery_fold_summary.csv", index=False)

# Keep the original notebook's compact output filename for compatibility.
pd.DataFrame({
    "metric": ["fraction_recovered_at_0.5", "confirmed_recall_at_0.5",
               "confirmed_total", "confirmed_recovered",
               "oof_roc_auc", "oof_average_precision"],
    "value": [
        float(row_05["confirmed_recall"]),
        float(row_05["confirmed_recall"]),
        int(row_05["confirmed_total"]),
        int(row_05["confirmed_recovered"]),
        float(oof_roc_auc),
        float(oof_average_precision)
    ]
}).to_csv("path2_recovery.csv", index=False)

print("Saved:")
for filename in [
    "path2_recovery.csv",
    "path2_recovery_metrics.csv",
    "path2_recovery_thresholds.csv",
    "path2_recovery_fold_summary.csv",
    "path2_recovery_oof_predictions.csv"
]:
    print(" -", filename)

## 8. Interpretation and limitations

- These are cross-validated predictions for rows whose catalog disposition is already resolved.
- Candidate entries are excluded; this analysis does not determine their true status.
- GroupKFold prevents the same host star from appearing in both training and test data within a fold.
- The model uses an explicit set of physical/observational features and excludes catalog disposition flags and scores.
- Requiring non-missing `koi_prad` preserves the original notebook's sample restriction; results therefore apply only to that restricted sample.
- A high confirmed-class probability is a model output, not independent evidence that a candidate is a planet.
- This notebook does **not** calculate planet occurrence rates or correct for detection completeness, transit probability, or catalog selection effects.

For ISEF reporting, describe the main result as **host-star-grouped out-of-fold sensitivity for catalog-confirmed entries**, and report the sample definition and threshold alongside the value.